### Hypergraph Coloring with DualHeadNet (Cooking200 Dataset)
This example solves hypergraph coloring on the Hypergraph_cooking200 dataset using the DualHeadNet architecture with GraphSAGE layers.

In [1]:
import sys
from pathlib import Path
current = Path.cwd()
root_path = current.parents[3]  
sys.path.append(str(root_path))
import torch
import dhg
from src import DualHeadNet, Datasets, Layer, LayerType
from src import init, get_device, from_file_to_hypergraph, run_pubo

/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Initialization and model setup:

In [2]:
init(cuda_index=0, reproducibility=True)
data_path = Datasets.Hypergraph_cooking200.path
hg = from_file_to_hypergraph(data_path, True).to(get_device())
init_dim = 1024
layers = [
    [Layer(LayerType.GRAPHSAGE, init_dim, 512, hidden_channels=512, num_layers=2, jk="last", drop_rate=0)],
    [],
    [Layer(LayerType.LINEAR, 512, 5, use_bn=True, dropout=0.2)],
    [Layer(LayerType.LINEAR, 512, 5, use_bn=True, dropout=0)],
]
x = torch.rand(hg.num_v, init_dim)
net = DualHeadNet(layers[0], layers[1], layers[2], layers[3])
gini_cons_lambda = lambda e, n: (-200 + 1 * e) / 1000
obj_cof_lambda = lambda e, n: 250 - e / 100
obj_cons_cof_lambda = lambda e, n: 100 - e / 100
cons_cof_lambda = lambda e, n:  5 + e / 1000

[WARNING] You have enabled the reproducibility feature, which uses a deterministic non-optimized algorithm, greatly affecting the running efficiency
[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 0)


Hypergraph(num_v=7403, num_e=2750)


#### Run optimization:

In [3]:
loss, outs,res = run_pubo(
    "coloring",
    net,
    x,
    hg,
    500,
    4e-4,
    "AdamW",
    clip_grad=True,
    simple=True,
    gini_cons_cof_lambda=gini_cons_lambda,
    obj_cof_lambda=obj_cof_lambda,
    cons_cof_lambda=cons_cof_lambda,
    obj_cons_cof_lambda=obj_cons_cof_lambda,
)


[INFO] INFO: 0 loops found
[INFO] Hypergraph to graph: 7403 vertices 2715 graph edges
  0%|          | 0/500 [00:00<?, ?it/s]/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/torch_geometric/utils/scatter.py:93: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(f"The usage of `scatter(reduce='{reduce}')` "
/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/torch_geometric/utils/scatter.py:98: UserWarning: scatter_reduce_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /home/builder/cbouss/pytorch/croot/pytorch_1685629640362/work/aten/src/ATen/Context.cpp:82.)
  return src.new_zeros(size).scatter_reduce_(
 41%|████      | 206/500 [00:

Epoch: 200 | coloring Loss: 0.05 | K Loss: 5.00 | obj cons: 0.00 | gini cons Loss: 5267.31


 81%|████████  | 406/500 [00:08<00:02, 45.10it/s]

Epoch: 400 | coloring Loss: 0.00 | K Loss: 4.00 | obj cons: 0.00 | gini cons Loss: 0.35


100%|██████████| 500/500 [00:10<00:00, 46.71it/s]


+------------[Evaluation Result]------------+
Qualified Edge: 2750/2750 (100.0%)
Color distribution:
[1542    0 4124 1347  390]
Not converged: 0
Num_Color_used: 4
+-------------------------------------------+


### Hypergraph Coloring with DualHeadNet (High School Dataset)
This example solves hypergraph coloring on the Hypergraph_high dataset using the same DualHeadNet architecture.


#### Initialization and model setup:


In [4]:
init(cuda_index=0, reproducibility=True)
data_path = Datasets.Hypergraph_high.path
hg = from_file_to_hypergraph(data_path, True).to(get_device())
g = dhg.Graph.from_hypergraph_clique(hg)
init_dim = 2048
layers = [
    [
    Layer(LayerType.GRAPHSAGE, init_dim, 256, hidden_channels=512, num_layers=2, jk="last", drop_rate=0)
    ],
    [],
    [Layer(LayerType.LINEAR, 256, 30, use_bn=True, dropout=0)],
    [Layer(LayerType.LINEAR, 256, 30, use_bn=True, dropout=0)],
]
x = torch.rand(hg.num_v, init_dim)
net = DualHeadNet(layers[0], layers[1], layers[2], layers[3])
gini_cons_lambda = lambda e, n: (-200 + 0.25 * e) / 1000
obj_cof_lambda = lambda e, n: 200 - e / 100  
obj_cons_cof_lambda = lambda e, n:  100 - e / 100 
cons_cof_lambda = lambda e, n: 10 + e / 100

[WARNING] You have enabled the reproducibility feature, which uses a deterministic non-optimized algorithm, greatly affecting the running efficiency
[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 0)


Hypergraph(num_v=327, num_e=7818)


#### Run optimization:

In [6]:
loss, outs,res = run_pubo(
    "coloring",
    net,
    x,
    hg,
    800,
    3e-4,
    "AdamW",
    clip_grad=True,
    gini_cons_cof_lambda=gini_cons_lambda,
    obj_cof_lambda=obj_cof_lambda,
    cons_cof_lambda=cons_cof_lambda,
    obj_cons_cof_lambda=obj_cons_cof_lambda,
)

[INFO] INFO: 0 loops found
[INFO] Hypergraph to graph: 327 vertices 5818 graph edges
  0%|          | 0/800 [00:00<?, ?it/s]/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/torch_geometric/utils/scatter.py:93: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(f"The usage of `scatter(reduce='{reduce}')` "
/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/torch_geometric/utils/scatter.py:98: UserWarning: scatter_reduce_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /home/builder/cbouss/pytorch/croot/pytorch_1685629640362/work/aten/src/ATen/Context.cpp:82.)
  return src.new_zeros(size).scatter_reduce_(
 26%|██▌       | 205/800 [00:0

Epoch: 200 | coloring Loss: 2.00 | K Loss: 21.00 | obj cons: 0.00 | gini cons Loss: 6.50


 51%|█████     | 405/800 [00:09<00:08, 45.72it/s]

Epoch: 400 | coloring Loss: 2.00 | K Loss: 21.00 | obj cons: 0.00 | gini cons Loss: 6.50


 76%|███████▌  | 605/800 [00:13<00:04, 45.37it/s]

Epoch: 600 | coloring Loss: 2.00 | K Loss: 21.00 | obj cons: 0.00 | gini cons Loss: 6.50


100%|██████████| 800/800 [00:17<00:00, 44.73it/s]


Epoch: 800 | coloring Loss: 2.00 | K Loss: 21.00 | obj cons: 0.00 | gini cons Loss: 6.50
+------------[Evaluation Result]------------+
Qualified Edge: 7816/7818 (100.0%)
Color distribution:
[ 0 14 22  0 19  0 10  0  0 11 21  5 16  6  9  0 26 15  0 20 29 19 16 18
  0  0 14 12 17  8]
Not converged: 13
Num_Color_used: 21
+-------------------------------------------+
